# GPU-Accelerated SVM Pipeline
**Parallel to `ml_pipeline.ipynb` — optimized for local NVIDIA GPU via RAPIDS cuML.**

Requires: CUDA 11.8+, RAPIDS 24.x (`cuml`, `cudf`)  
Install: https://rapids.ai/start.html — use the selector for your CUDA version.

Falls back gracefully to scikit-learn if cuML is not available.

### Feature set
| Source | Features |
|--------|----------|
| Lexical (URL) | `host_length`, `path_depth`, `dot_count`, `hyphen_count`, `digit_ratio`, `subdomain_depth`, `full_length`, `digit_count`, `has_at`, `has_ip` |
| Semantic (embeddings) | `score_1`–`score_5` (cosine sim to nearest phish/tranco matches) |

In [ ]:
# Check GPU and RAPIDS availability
import subprocess, sys

try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode()
    print('GPU detected:', gpu_info.strip())
except FileNotFoundError:
    print('nvidia-smi not found — no GPU detected')

try:
    import cuml
    import cudf
    USE_GPU = True
    print(f'cuML {cuml.__version__} available — using GPU path')
except ImportError:
    USE_GPU = False
    print('cuML not found — falling back to scikit-learn (CPU)')
    print('To install RAPIDS: https://rapids.ai/start.html')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from urllib.parse import urlparse
import re
import time

if USE_GPU:
    from cuml.svm import LinearSVC
    from cuml.preprocessing import StandardScaler
    from cuml.metrics import confusion_matrix
    from cuml.model_selection import train_test_split
else:
    from sklearn.svm import LinearSVC
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import confusion_matrix
    from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report  # always use sklearn for reporting

## 1. Load Data

In [ ]:
phish = pd.read_csv('../data/processed_input/phishtank_cleaned.csv')
tranco = pd.read_csv('../data/processed_input/tranco_cleaned.csv')

top5_phish = pd.read_csv('../data/summarized_output/top5_per_phish.csv')
top5_tranco = pd.read_csv('../data/summarized_output/top5_per_tranco.csv')

print(f'PhishTank: {len(phish):,} rows')
print(f'Tranco:    {len(tranco):,} rows')
print(f'top5_phish: {top5_phish.shape}  cols: {list(top5_phish.columns)}')
print(f'top5_tranco: {top5_tranco.shape}  cols: {list(top5_tranco.columns)}')

## 2. Feature Engineering

In [ ]:
IP_RE = re.compile(r'\d{1,3}(\.\d{1,3}){3}')

def extract_features(df, url_col='url'):
    d = df.copy()
    parsed = d[url_col].apply(lambda x: urlparse(str(x)))
    host   = parsed.apply(lambda p: p.netloc.lower())
    path   = parsed.apply(lambda p: p.path)

    d['host_length']     = host.str.len()
    d['full_length']     = d[url_col].str.len()
    d['path_depth']      = path.apply(lambda p: p.strip('/').count('/') + 1 if p.strip('/') else 0)
    d['dot_count']       = d[url_col].str.count(r'\.')
    d['hyphen_count']    = d[url_col].str.count('-')
    d['digit_count']     = d[url_col].str.count(r'\d')
    d['digit_ratio']     = d['digit_count'] / d['full_length'].replace(0, np.nan)
    d['subdomain_depth'] = host.str.count(r'\.')
    d['has_at']          = d[url_col].str.contains('@').astype(int)
    d['has_ip']          = host.apply(lambda h: int(bool(IP_RE.search(h))))

    return d

phish  = extract_features(phish)
tranco = extract_features(tranco)

phish['label']  = 1
tranco['label'] = 0

print('Feature extraction done.')
print(phish[['url','host_length','path_depth','dot_count','has_ip','label']].head(3))

## 3. Merge Cosine Similarity Scores
The top5 CSVs contain pre-computed embedding similarity scores — these encode *semantic* distance to known legitimate domains, which is a strong phishing signal.

In [ ]:
score_cols = ['score_1', 'score_2', 'score_3', 'score_4', 'score_5']

# top5 CSVs use 'item' as the URL key
phish_scores  = top5_phish[['item'] + score_cols].rename(columns={'item': 'url'})
tranco_scores = top5_tranco[['item'] + score_cols].rename(columns={'item': 'url'})

# Normalize URL representation before joining
def strip_url(s):
    return s.str.strip().str.rstrip('/')

phish['url_key']       = strip_url(phish['url'])
phish_scores['url_key'] = strip_url(phish_scores['url'])
tranco['url_key']       = strip_url(tranco['url'] if 'url' in tranco.columns else tranco.iloc[:,0])
tranco_scores['url_key'] = strip_url(tranco_scores['url'])

phish  = phish.merge(phish_scores[['url_key'] + score_cols],  on='url_key', how='left')
tranco = tranco.merge(tranco_scores[['url_key'] + score_cols], on='url_key', how='left')

phish_match_rate  = phish[score_cols].notna().all(axis=1).mean()
tranco_match_rate = tranco[score_cols].notna().all(axis=1).mean()
print(f'Score merge coverage — phish: {phish_match_rate:.1%}  tranco: {tranco_match_rate:.1%}')

## 4. Build Training Dataset

In [ ]:
LEXICAL_FEATURES = [
    'host_length', 'full_length', 'path_depth', 'dot_count',
    'hyphen_count', 'digit_ratio', 'subdomain_depth', 'has_at', 'has_ip'
]
ALL_FEATURES = LEXICAL_FEATURES + score_cols

combined = pd.concat([phish, tranco], ignore_index=True)

# Drop rows missing any feature
before = len(combined)
combined = combined.dropna(subset=ALL_FEATURES)
print(f'Dropped {before - len(combined):,} rows with NaN features ({len(combined):,} remaining)')
print(f'Class balance — phish: {combined["label"].sum():,}  legit: {(combined["label"]==0).sum():,}')

X = combined[ALL_FEATURES].astype(np.float32)
y = combined['label'].astype(np.int32)

print(f'\nFeature matrix: {X.shape}')

## 5. Train / Test Split + Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

## 6. Train LinearSVC

In [ ]:
if USE_GPU:
    # cuML wants cuDF arrays or numpy float32
    import cupy as cp
    X_tr = cp.array(X_train_scaled.values if hasattr(X_train_scaled, 'values') else X_train_scaled)
    y_tr = cp.array(y_train.values if hasattr(y_train, 'values') else y_train)
    X_te = cp.array(X_test_scaled.values if hasattr(X_test_scaled, 'values') else X_test_scaled)
    y_te = cp.array(y_test.values if hasattr(y_test, 'values') else y_test)
else:
    X_tr, y_tr = X_train_scaled, y_train
    X_te, y_te = X_test_scaled, y_test

svm = LinearSVC(C=1.0, max_iter=2000)

t0 = time.time()
svm.fit(X_tr, y_tr)
elapsed = time.time() - t0

print(f'Training complete in {elapsed:.2f}s  (backend: {"cuML GPU" if USE_GPU else "sklearn CPU"})')

## 7. Evaluation

In [ ]:
y_pred = svm.predict(X_te)

# Convert back to numpy if on GPU
if USE_GPU:
    y_pred_np = cp.asnumpy(y_pred)
    y_te_np   = cp.asnumpy(y_te)
else:
    y_pred_np = np.array(y_pred)
    y_te_np   = np.array(y_te)

print(classification_report(y_te_np, y_pred_np, target_names=['Legitimate', 'Phishing']))

In [ ]:
cm = confusion_matrix(y_te_np, y_pred_np)
if USE_GPU:
    cm = cp.asnumpy(cm)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: Legit', 'Pred: Phish'],
            yticklabels=['True: Legit', 'True: Phish'], ax=ax)
ax.set_title('Confusion Matrix — LinearSVC')
plt.tight_layout()
plt.show()

## 8. Feature Importance
LinearSVC coefficients are directly interpretable as feature weights.

In [ ]:
coef = svm.coef_
if USE_GPU:
    coef = cp.asnumpy(coef)
coef = np.array(coef).flatten()

importance_df = pd.DataFrame({'feature': ALL_FEATURES, 'weight': coef})
importance_df = importance_df.reindex(importance_df['weight'].abs().sort_values(ascending=False).index)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if w > 0 else '#2ecc71' for w in importance_df['weight']]
ax.barh(importance_df['feature'], importance_df['weight'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('SVM Weight  (positive → phishing signal)')
ax.set_title('Feature Importance — LinearSVC')
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

## 9. Lexical-Only vs Lexical+Semantic Comparison
Ablation: how much do the cosine similarity scores add over pure lexical features?

In [ ]:
from sklearn.svm import LinearSVC as SklearnSVC
from sklearn.preprocessing import StandardScaler as SklearnScaler
from sklearn.model_selection import train_test_split as sklearn_split
from sklearn.metrics import f1_score

results = {}

for label, feats in [('Lexical only', LEXICAL_FEATURES), ('Lexical + Semantic', ALL_FEATURES)]:
    Xf = combined[feats].astype(np.float32)
    yf = combined['label'].astype(np.int32)
    Xf_tr, Xf_te, yf_tr, yf_te = sklearn_split(Xf, yf, test_size=0.2, random_state=42, stratify=yf)
    sc = SklearnScaler()
    Xf_tr_s = sc.fit_transform(Xf_tr)
    Xf_te_s = sc.transform(Xf_te)
    clf = SklearnSVC(C=1.0, max_iter=2000)
    clf.fit(Xf_tr_s, yf_tr)
    preds = clf.predict(Xf_te_s)
    results[label] = {
        'F1 (macro)': f1_score(yf_te, preds, average='macro'),
        'F1 (phishing)': f1_score(yf_te, preds, pos_label=1),
    }

pd.DataFrame(results).T.round(4)

## 10. Save Model

In [ ]:
import joblib, os

os.makedirs('../models', exist_ok=True)

# Always save sklearn-compatible model for portability
joblib.dump({'scaler': scaler, 'svm': svm, 'features': ALL_FEATURES}, '../models/linear_svc.joblib')
print('Model saved to ../models/linear_svc.joblib')